# Ejercicio 3 - Consultas directas sobre archivos Parquet

Exploracion inicial de los Parquet de taxis amarillos y verdes **sin importarlos a una tabla**: cada consulta usa `read_parquet(...)`, `glob(...)`, `parquet_file_metadata(...)` o `parquet_schema(...)` directamente sobre `data/raw/`.

Las consultas viven en `sql/ejercicio3/` (una por archivo, con su objetivo y fuente en el encabezado). Este notebook las ejecuta y muestra; la documentacion completa (SQL + resultado + tiempo) se genera con:

```bash
python scripts/run_sql.py ejercicio3   # -> docs/resultados/ejercicio3.md
```

Las decisiones tomadas a partir de cada resultado estan en `docs/ejercicio3_exploracion.md`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> scripts/
from lab import conectar, ejecutar_sql, consulta, RAIZ_REPO

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def correr(archivo):
    """Ejecuta un .sql, muestra sus metadatos y tiempo, y devuelve el DataFrame."""
    meta, df, seg = ejecutar_sql(con, archivo)
    print(f"[{meta.get('id')}] {meta.get('titulo', '')}")
    for clave in ("pregunta", "objetivo", "fuente"):
        if clave in meta:
            print(f"  {clave}: {meta[clave]}")
    print(f"  tiempo: {seg:.3f} s | filas: {len(df)}")
    return df

con = conectar(vistas=False)   # sin vistas: todo se lee directo de los Parquet

## 3.1 Cantidad de archivos

In [ ]:
correr("sql/ejercicio3/3_01_cantidad_archivos.sql")

## 3.2 Cantidad de registros

Primero con los **metadatos** del footer de cada Parquet (no lee datos), luego con un `count(*)` sobre los archivos. Ambos totales deben coincidir.

In [ ]:
por_archivo = correr("sql/ejercicio3/3_02_registros_por_archivo.sql")
por_archivo

In [ ]:
totales = correr("sql/ejercicio3/3_03_registros_totales.sql")
print("Suma de metadatos por tipo:")
print(por_archivo.groupby("tipo")["registros"].sum())
totales

## 3.3 Columnas presentes

In [ ]:
correr("sql/ejercicio3/3_04_columnas_yellow.sql")[["column_name", "column_type", "null"]]

In [ ]:
correr("sql/ejercicio3/3_05_columnas_green.sql")[["column_name", "column_type", "null"]]

In [ ]:
correr("sql/ejercicio3/3_06_comparacion_columnas.sql")

## 3.4 Tipos de datos por archivo

Las filas con `n_tipos > 1` o `en_archivos < archivos_totales` aparecen primero: indican cambios de esquema entre archivos.

In [ ]:
tipos = correr("sql/ejercicio3/3_07_tipos_por_archivo.sql")
tipos[(tipos.n_tipos > 1) | (tipos.en_archivos < tipos.archivos_totales)]

In [ ]:
tipos

## 3.5 Muestra de registros

In [ ]:
correr("sql/ejercicio3/3_08_muestra_yellow.sql")

In [ ]:
correr("sql/ejercicio3/3_09_muestra_green.sql")

## 3.6 Problemas de calidad de datos

`SUMMARIZE` da un perfil de cada columna (min, max, cuartiles, % de nulos). Luego se cuantifican reglas concretas.

In [ ]:
resumen_y = correr("sql/ejercicio3/3_10_resumen_yellow.sql")
resumen_y[["column_name", "column_type", "min", "max", "avg", "q50", "approx_unique", "null_percentage"]]

In [ ]:
resumen_g = correr("sql/ejercicio3/3_11_resumen_green.sql")
resumen_g[["column_name", "column_type", "min", "max", "avg", "q50", "approx_unique", "null_percentage"]]

In [ ]:
reglas = correr("sql/ejercicio3/3_12_reglas_calidad.sql")
# porcentaje sobre el total de cada tipo
pct = reglas.set_index("tipo")
(pct.drop(columns="registros").div(pct["registros"], axis=0) * 100).round(3).T

In [ ]:
correr("sql/ejercicio3/3_13_fechas_fuera_de_mes.sql")

In [ ]:
correr("sql/ejercicio3/3_14_codigos_categoricos.sql")

In [ ]:
correr("sql/ejercicio3/3_15_consistencia_montos.sql")

In [ ]:
correr("sql/ejercicio3/3_16_duplicados.sql")